# CFP Ranker '26: Data Collection
This notebook maintains one SQLite database, `espn_cfb_2026.db`, as the **source of truth** for raw ESPN data. Running it again only calls ESPN for data that is missing or may have changed. Everything derived from this data (team defensive stats, rollups, rankings) is computed from the database in later notebooks and is not stored here.

## When each kind of data is fetched

| Data | Fetched when |
| --- | --- |
| FBS teams | The `Teams` table has no FBS teams yet (or `force=True`) |
| Daily scoreboard | The day was never fetched; or it is within the last 7 / next 14 days and was last fetched over 24 hours ago; or it was fetched before that day's games could have finished |
| Game box score | A Final game has no stored box score; or ESPN had no box score yet and the last check was over 24 hours ago; or the game kicked off within the last 7 days and the last check was over 24 hours ago |

A re-checked box score is only rewritten if ESPN's data actually changed (compared by hash).

## Tables

| Table | Purpose |
| --- | --- |
| `Teams` | FBS teams plus any FCS opponents seen in games |
| `Games` | Every scoreboard event (any status) |
| `Box_Scores` | One row per game, team and statistic, parsed from the raw payloads |
| `Raw_Boxscores` | The raw ESPN `boxscore` JSON per game (compressed), its hash and when it was fetched. `Box_Scores` can be rebuilt from this with no network calls |
| `Scoreboard_Fetches` | Which days of the scoreboard have been fetched, and when |

**First run on an existing database:** raw payloads were not stored before, so every Final game is fetched once (about 700 requests) to fill `Raw_Boxscores`. After that, runs make few or no requests.

## 1. Setup

In [7]:
import hashlib
import json
import sqlite3
import time
import zlib
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from datetime import time as dtime
 
import pandas as pd
import requests
 
# --- Season and database ---
YEAR = 2026
DB_FILE = f"espn_cfb_{YEAR}.db"
SEASON_START = date(YEAR, 8, 1)
SEASON_END = date(YEAR + 1, 2, 1)
 
# --- Refresh rules ---
RECENT_DAYS = 7            # games/days this recent are re-checked for revisions
LOOKAHEAD_DAYS = 14        # upcoming days whose schedules are kept fresh
REFRESH_AFTER_HOURS = 24   # minimum time between re-checks of the same item
REQUEST_DELAY = 0.15       # seconds to wait after each API call
 
# --- ESPN endpoints ---
SCOREBOARD_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"
SUMMARY_URL = "https://site.api.espn.com/apis/site/v2/sports/football/college-football/summary"
STANDINGS_URL = "https://site.web.api.espn.com/apis/v2/sports/football/college-football/standings"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json",
}

_session = requests.Session()
_session.headers.update(HEADERS)
api_calls = 0  # every network request goes through api_get, so this counts them all

def utcnow():
    return datetime.now(timezone.utc)
 
def iso(dt):
    return dt.isoformat(timespec="seconds")

def parse_espn_date(value):
    """Parse ESPN timestamps like '2026-08-29T16:00Z' into timezone-aware datetimes."""
    ts = pd.Timestamp(value)
    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    return ts.to_pydatetime()

def api_get(url, params=None, retries=3):
    """GET JSON from ESPN with simple retries. Raises requests.RequestException if all attempts fail."""
    global api_calls
    last_exc = None
    for attempt in range(retries):
        api_calls += 1
        try:
            response = _session.get(url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()
            time.sleep(REQUEST_DELAY)
            return data
        except requests.RequestException as exc:
            last_exc = exc
            time.sleep(1.5 * (attempt + 1))
    raise last_exc

## 2. Database

In [8]:
conn = sqlite3.connect(DB_FILE)
conn.execute("PRAGMA foreign_keys = ON")
 
conn.executescript("""
CREATE TABLE IF NOT EXISTS Teams (
    team_id TEXT PRIMARY KEY,
    display_name TEXT NOT NULL,
    short_name TEXT,
    conference TEXT,
    conference_id TEXT
);
CREATE TABLE IF NOT EXISTS Games (
    game_id TEXT PRIMARY KEY,
    game_date TEXT,
    name TEXT,
    week INTEGER,
    home_team_id TEXT REFERENCES Teams(team_id),
    away_team_id TEXT REFERENCES Teams(team_id),
    home_score INTEGER,
    away_score INTEGER,
    status TEXT,
    venue TEXT,
    neutral_site INTEGER
);
CREATE TABLE IF NOT EXISTS Box_Scores (
    game_id TEXT NOT NULL REFERENCES Games(game_id),
    team_id TEXT NOT NULL REFERENCES Teams(team_id),
    stat_name TEXT NOT NULL,
    stat_label TEXT,
    stat_value REAL,
    display_value TEXT,
    PRIMARY KEY (game_id, team_id, stat_name)
);
-- Raw ESPN "boxscore" section per game. payload is zlib-compressed JSON.
-- payload/payload_hash are NULL when ESPN had not published a box score at fetch time.
CREATE TABLE IF NOT EXISTS Raw_Boxscores (
    game_id TEXT PRIMARY KEY REFERENCES Games(game_id),
    fetched_at TEXT NOT NULL,
    payload_hash TEXT,
    payload BLOB
);
CREATE TABLE IF NOT EXISTS Scoreboard_Fetches (
    game_date TEXT PRIMARY KEY,   -- YYYY-MM-DD
    fetched_at TEXT NOT NULL
);
CREATE INDEX IF NOT EXISTS idx_box_scores_team ON Box_Scores(team_id);
CREATE INDEX IF NOT EXISTS idx_games_date ON Games(game_date);
""")
conn.commit()
print(f"SQLite database ready: {DB_FILE}")

SQLite database ready: espn_cfb_2026.db


## 3. Teams

Fetched once. FCS opponents are added automatically when they appear in the schedule.

In [10]:
def parse_standings_node(node, parent_conference="", parent_conf_id=""):
    """Recursively walk ESPN's standings tree and return teams with their conference name and ID."""
    teams = []
    current_name = node.get("name", "")
    current_id = str(node.get("id", ""))
 
    # Top-level league labels must not override the real conference name/ID
    is_root_label = current_name.lower() in ["fbs", "ncaa - football bowl subdivision", "football bowl subdivision"]
    if is_root_label:
        conf_label, conf_id = "", ""
    else:
        conf_label = parent_conference if parent_conference else current_name
        conf_id = parent_conf_id if parent_conf_id else current_id
 
    for entry in node.get("standings", {}).get("entries", []):
        team = entry.get("team", {})
        if team:
            teams.append({
                "team_id": team.get("id"),
                "display_name": team.get("displayName"),
                "short_name": team.get("shortDisplayName"),
                "conference": conf_label,
                "conference_id": conf_id,
            })
 
    for child in node.get("children", []):
        teams.extend(parse_standings_node(child, parent_conference=conf_label, parent_conf_id=conf_id))
    return teams

def ensure_team(team_id, display_name):
    """Make sure a team row exists (needed for foreign keys). Never overwrites an existing row."""
    if team_id is not None:
        conn.execute(
            "INSERT OR IGNORE INTO Teams (team_id, display_name) VALUES (?, ?)",
            (str(team_id), display_name or "Unknown"),
        )

def load_fbs_teams(force=False):
    stored = conn.execute(
        "SELECT COUNT(*) FROM Teams WHERE conference IS NOT NULL AND conference != ''"
    ).fetchone()[0]
    if stored and not force:
        print(f"Teams: {stored} FBS teams already stored, no fetch needed.")
        return
 
    data = api_get(STANDINGS_URL, params={"group": "80"})  # 80 = FBS
    teams = {t["team_id"]: t for t in parse_standings_node(data) if t["team_id"]}
    conn.executemany(
        """INSERT INTO Teams (team_id, display_name, short_name, conference, conference_id)
           VALUES (?, ?, ?, ?, ?)
           ON CONFLICT(team_id) DO UPDATE SET
               display_name = excluded.display_name, short_name = excluded.short_name,
               conference = excluded.conference, conference_id = excluded.conference_id""",
        [(str(t["team_id"]), t["display_name"], t["short_name"],
          t["conference"] or None, t["conference_id"] or None) for t in teams.values()],
    )
    conn.commit()
    print(f"Teams: loaded {len(teams)} FBS teams.")

## 4. Schedule (Daily Scoreboard)

In [11]:
GAME_COLUMNS = ["game_id", "game_date", "name", "week", "home_team_id", "away_team_id",
                "home_score", "away_score", "status", "venue", "neutral_site"]
 
GAME_UPSERT = """
INSERT INTO Games (game_id, game_date, name, week, home_team_id, away_team_id,
                   home_score, away_score, status, venue, neutral_site)
VALUES (:game_id, :game_date, :name, :week, :home_team_id, :away_team_id,
        :home_score, :away_score, :status, :venue, :neutral_site)
ON CONFLICT(game_id) DO UPDATE SET
    game_date = excluded.game_date, name = excluded.name, week = excluded.week,
    home_team_id = excluded.home_team_id, away_team_id = excluded.away_team_id,
    home_score = excluded.home_score, away_score = excluded.away_score,
    status = excluded.status, venue = excluded.venue, neutral_site = excluded.neutral_site
"""

def int_or_none(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None

def parse_event(event):
    competition = (event.get("competitions") or [{}])[0]
    competitors = competition.get("competitors", [])
    home = next((c for c in competitors if c.get("homeAway") == "home"), {})
    away = next((c for c in competitors if c.get("homeAway") == "away"), {})
    home_team, away_team = home.get("team", {}), away.get("team", {})
    neutral = competition.get("neutralSite")
    return {
        "game_id": str(event.get("id")),
        "game_date": event.get("date"),
        "name": event.get("name"),
        "week": int_or_none((event.get("week") or {}).get("number")),
        "home_team_id": str(home_team["id"]) if home_team.get("id") is not None else None,
        "home_team": home_team.get("displayName"),
        "away_team_id": str(away_team["id"]) if away_team.get("id") is not None else None,
        "away_team": away_team.get("displayName"),
        "home_score": int_or_none(home.get("score")),
        "away_score": int_or_none(away.get("score")),
        "status": ((event.get("status") or {}).get("type") or {}).get("description"),
        "venue": (competition.get("venue") or {}).get("fullName"),
        "neutral_site": None if neutral is None else int(bool(neutral)),
    }

def scoreboard_dates_to_fetch():
    """Days whose scoreboard is missing or may have changed."""
    now = utcnow()
    today = now.date()
    last_day = min(SEASON_END, today + timedelta(days=LOOKAHEAD_DAYS))
    recent_start = today - timedelta(days=RECENT_DAYS)
    stale_before = now - timedelta(hours=REFRESH_AFTER_HOURS)
    fetched = {d: datetime.fromisoformat(t)
               for d, t in conn.execute("SELECT game_date, fetched_at FROM Scoreboard_Fetches")}
 
    dates, day = [], SEASON_START
    while day <= last_day:
        seen = fetched.get(day.isoformat())
        if seen is None:
            dates.append(day)  # never fetched
        else:
            # Late games end by about noon UTC the next day; anything fetched before that may be incomplete
            games_over_by = datetime.combine(day + timedelta(days=1), dtime(12, 0), tzinfo=timezone.utc)
            if seen < stale_before and (day >= recent_start or seen < games_over_by):
                dates.append(day)
        day += timedelta(days=1)
    return dates

def refresh_schedule():
    stats = Counter()
    for day in scoreboard_dates_to_fetch():
        try:
            data = api_get(SCOREBOARD_URL, params={
                "dates": day.strftime("%Y%m%d"), "groups": "80", "limit": "500"})
        except requests.RequestException:
            stats["failed_days"] += 1  # not recorded, so it is retried next run
            continue
 
        for event in data.get("events", []):
            game = parse_event(event)
            ensure_team(game["home_team_id"], game["home_team"])
            ensure_team(game["away_team_id"], game["away_team"])
 
            old = conn.execute(
                "SELECT " + ", ".join(GAME_COLUMNS[1:]) + " FROM Games WHERE game_id = ?",
                (game["game_id"],)).fetchone()
            new = tuple(game[c] for c in GAME_COLUMNS[1:])
            if old is None:
                stats["new_games"] += 1
            elif old != new:
                stats["changed_games"] += 1
            conn.execute(GAME_UPSERT, game)
 
        conn.execute(
            """INSERT INTO Scoreboard_Fetches (game_date, fetched_at) VALUES (?, ?)
               ON CONFLICT(game_date) DO UPDATE SET fetched_at = excluded.fetched_at""",
            (day.isoformat(), iso(utcnow())))
        conn.commit()
        stats["days_fetched"] += 1
 
    print("Schedule:", ", ".join(f"{k}={stats[k]}" for k in
          ["days_fetched", "new_games", "changed_games", "failed_days"]))
    return stats

## 5. Box Scores
ESPN's `boxscore` section is stored raw (compressed) in `Raw_Boxscores`, and parsed into `Box_Scores`. Parsing is a separate step, so a parsing fix only needs `rebuild_box_scores_from_raw()` and no new requests.

`stat_value` uses ESPN's numeric `value` when it has one, and otherwise falls back to a numeric `displayValue`. Compound values such as `5-12` (made-attempts) or `31:22` stay `NULL` in `stat_value`; the original text is always in `display_value`.

In [12]:
BOXSCORE_INSERT = """
INSERT INTO Box_Scores (game_id, team_id, stat_name, stat_label, stat_value, display_value)
VALUES (?, ?, ?, ?, ?, ?)
ON CONFLICT(game_id, team_id, stat_name) DO UPDATE SET
    stat_label = excluded.stat_label, stat_value = excluded.stat_value,
    display_value = excluded.display_value
"""

RAW_UPSERT = """
INSERT INTO Raw_Boxscores (game_id, fetched_at, payload_hash, payload) VALUES (?, ?, ?, ?)
ON CONFLICT(game_id) DO UPDATE SET
    fetched_at = excluded.fetched_at, payload_hash = excluded.payload_hash, payload = excluded.payload
"""

def pack_boxscore(boxscore):
    """Return (hash, compressed bytes) for a boxscore dict. The hash is of a canonical JSON form."""
    canonical = json.dumps(boxscore, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(canonical).hexdigest(), zlib.compress(canonical, 9)

def unpack_boxscore(blob):
    return json.loads(zlib.decompress(blob))

def numeric_or_none(value):
    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def parse_team_stats(game_id, boxscore):
    """Flatten a boxscore dict into Box_Scores rows."""
    rows = []
    for team in boxscore.get("teams", []):
        team_id = team.get("team", {}).get("id")
        if team_id is None:
            continue
        for stat in team.get("statistics", []):
            stat_name = stat.get("name") or stat.get("label")
            if not stat_name:
                continue
            value = numeric_or_none(stat.get("value"))
            if value is None:
                value = numeric_or_none(stat.get("displayValue"))
            rows.append((game_id, str(team_id), str(stat_name), stat.get("label"),
                         value, stat.get("displayValue")))
    return rows

def store_boxscore(game_id, boxscore, payload_hash, blob, fetched_at):
    """Write the raw payload and its parsed rows together (the caller commits)."""
    for team in boxscore.get("teams", []):
        info = team.get("team", {})
        ensure_team(info.get("id"), info.get("displayName"))
    conn.execute("DELETE FROM Box_Scores WHERE game_id = ?", (game_id,))
    conn.executemany(BOXSCORE_INSERT, parse_team_stats(game_id, boxscore))
    conn.execute(RAW_UPSERT, (game_id, fetched_at, payload_hash, blob))

def rebuild_box_scores_from_raw():
    """Re-parse every stored payload into Box_Scores. No network calls."""
    rows = conn.execute("SELECT game_id, payload FROM Raw_Boxscores WHERE payload IS NOT NULL").fetchall()
    for game_id, blob in rows:
        conn.execute("DELETE FROM Box_Scores WHERE game_id = ?", (game_id,))
        conn.executemany(BOXSCORE_INSERT, parse_team_stats(game_id, unpack_boxscore(blob)))
    conn.commit()
    print(f"Rebuilt Box_Scores from {len(rows)} stored payloads.")

In [14]:
def final_games_with_fetch_state():
    cur = conn.execute("""
        SELECT g.game_id, g.game_date, r.fetched_at, r.payload_hash
        FROM Games g
        LEFT JOIN Raw_Boxscores r ON r.game_id = g.game_id
        WHERE g.status = 'Final'""")
    cols = [c[0] for c in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]

def fetch_reason(game, now):
    """Why this Final game needs a fetch, or None to skip it."""
    if game["fetched_at"] is None:
        return "missing"
    stale = now - datetime.fromisoformat(game["fetched_at"]) >= timedelta(hours=REFRESH_AFTER_HOURS)
    if game["payload_hash"] is None:
        return "no_boxscore_yet" if stale else None
    if game["game_date"]:
        kicked_off = parse_espn_date(game["game_date"])
        if stale and now - kicked_off <= timedelta(days=RECENT_DAYS):
            return "recent_recheck"
    return None

def ingest_box_scores():
    stats = Counter()
    failed = []
    now = utcnow()
 
    todo = []
    for game in final_games_with_fetch_state():
        reason = fetch_reason(game, now)
        if reason:
            todo.append((game, reason))
        else:
            stats["skipped"] += 1
 
    for game, reason in todo:
        game_id = game["game_id"]
        try:
            summary = api_get(SUMMARY_URL, params={"event": game_id})
        except requests.RequestException as exc:
            failed.append((game_id, str(exc)))
            continue
 
        boxscore = summary.get("boxscore") or {}
        stamp = iso(utcnow())
        try:
            if not boxscore.get("teams"):
                # Nothing published yet. Record the check (keeping any existing payload) so we don't re-ask right away.
                conn.execute(
                    """INSERT INTO Raw_Boxscores (game_id, fetched_at) VALUES (?, ?)
                       ON CONFLICT(game_id) DO UPDATE SET fetched_at = excluded.fetched_at""",
                    (game_id, stamp))
                stats["unchanged" if game["payload_hash"] else "no_boxscore"] += 1
            else:
                payload_hash, blob = pack_boxscore(boxscore)
                if game["payload_hash"] == payload_hash:
                    conn.execute("UPDATE Raw_Boxscores SET fetched_at = ? WHERE game_id = ?", (stamp, game_id))
                    stats["unchanged"] += 1
                else:
                    store_boxscore(game_id, boxscore, payload_hash, blob, stamp)
                    stats["updated" if game["payload_hash"] else "new"] += 1
            conn.commit()
        except sqlite3.Error as exc:
            conn.rollback()
            failed.append((game_id, f"database error: {exc}"))
 
    print("Box scores:", ", ".join(f"{k}={stats[k]}" for k in
          ["new", "updated", "unchanged", "no_boxscore", "skipped"]) + f", failed={len(failed)}")
    if failed:
        print("Failed (retried automatically next run):", failed)
    stats["failed"] = len(failed)
    return stats

## 6. Update the database

Safe to run as often as you like. Prints how many ESPN requests the run made; a run with nothing new should make none.

In [16]:
def update_database(force_teams=False):
    start_calls = api_calls
    load_fbs_teams(force=force_teams)
    schedule_stats = refresh_schedule()
    box_stats = ingest_box_scores()
    calls = api_calls - start_calls
    print(f"ESPN requests this run: {calls}")
    return {"schedule": schedule_stats, "box_scores": box_stats, "api_calls": calls}
 
 
update_database()

Teams: 138 FBS teams already stored, no fetch needed.
Schedule: days_fetched=0, new_games=0, changed_games=0, failed_days=0
Box scores: new=0, updated=0, unchanged=0, no_boxscore=0, skipped=685, failed=0
ESPN requests this run: 0


{'schedule': Counter(),
 'box_scores': Counter({'skipped': 685, 'failed': 0}),
 'api_calls': 0}

## 7. Checks

In [17]:
final_not_stored = conn.execute("""
    SELECT COUNT(*) FROM Games g
    LEFT JOIN Raw_Boxscores r ON r.game_id = g.game_id
    WHERE g.status = 'Final' AND r.payload_hash IS NULL""").fetchone()[0]
print("Final games without a stored box score:", final_not_stored)
 
print("FBS teams with no games:", conn.execute("""
    SELECT t.display_name FROM Teams t
    WHERE t.conference IS NOT NULL AND t.conference != ''
      AND NOT EXISTS (SELECT 1 FROM Games g WHERE t.team_id IN (g.home_team_id, g.away_team_id))
""").fetchall())
 
print("Foreign key violations:", conn.execute("PRAGMA foreign_key_check").fetchall())
 
print("Row counts (Teams, Games, Box_Scores, Raw_Boxscores):", conn.execute("""
    SELECT (SELECT COUNT(*) FROM Teams), (SELECT COUNT(*) FROM Games),
           (SELECT COUNT(*) FROM Box_Scores), (SELECT COUNT(*) FROM Raw_Boxscores)""").fetchone())

Final games without a stored box score: 0
FBS teams with no games: []
Foreign key violations: []
Row counts (Teams, Games, Box_Scores, Raw_Boxscores): (238, 824, 20550, 685)


A stat should only have fewer numeric rows than total rows if it is a compound value (for example `completionAttempts`, `totalPenaltiesYards`) or, for `fourthDownEff`, a team with no attempts.

In [18]:
pd.read_sql_query("""
SELECT stat_name, COUNT(*) AS rows_, COUNT(stat_value) AS numeric_rows
FROM Box_Scores
GROUP BY stat_name
ORDER BY stat_name
""", conn)

,stat_name,rows_,numeric_rows
0,completionAttempts,1370,0
1,firstDowns,1370,1370
2,fourthDownEff,1370,1141
3,fumblesLost,1370,1370
4,interceptions,1370,1370
5,netPassingYards,1370,1370
6,possessionTime,1370,1370
7,rushingAttempts,1370,1370
8,rushingYards,1370,1370
9,thirdDownEff,1370,1370


## 8. Quick look

In [21]:
pd.read_sql_query("""
SELECT g.week, g.game_date, h.display_name AS home, g.home_score,
       a.display_name AS away, g.away_score
FROM Games g
JOIN Teams h ON h.team_id = g.home_team_id
JOIN Teams a ON a.team_id = g.away_team_id
WHERE g.status = 'Final'
ORDER BY g.game_date DESC
LIMIT 10
""", conn)

,week,game_date,home,home_score,away,away_score
0,10,2026-11-08T03:30Z,UNLV Rebels,35,Wyoming Cowboys,41
1,10,2026-11-08T03:30Z,Oregon State Beavers,21,Texas State Bobcats,42
2,10,2026-11-08T02:30Z,UTEP Miners,14,Hawai'i Rainbow Warriors,17
3,10,2026-11-08T02:30Z,Utah State Aggies,17,Fresno State Bulldogs,10
4,10,2026-11-08T02:30Z,Texas Tech Red Raiders,53,West Virginia Mountaineers,48
5,10,2026-11-08T01:00Z,New Mexico State Aggies,14,Liberty Flames,30
6,10,2026-11-08T00:45Z,Florida Gators,7,Oklahoma Sooners,23
7,10,2026-11-08T00:30Z,Army Black Knights,21,Air Force Falcons,24
8,10,2026-11-08T00:30Z,Michigan Wolverines,6,Michigan State Spartans,15
9,10,2026-11-08T00:30Z,Georgia Tech Yellow Jackets,17,Louisville Cardinals,44


## Maintenance

- **Parsing change:** edit `parse_team_stats`, then run `rebuild_box_scores_from_raw()`. No requests needed.
- **Force a team refresh:** `update_database(force_teams=True)`.
- **Remove the old derived table** from earlier experiments (derived stats are now computed on demand):

In [ ]:
# conn.execute("DROP TABLE IF EXISTS Defense_Stats"); conn.commit()
# rebuild_box_scores_from_raw()